In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import rioxarray
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.colors import ListedColormap, BoundaryNorm
from mpl_toolkits.axes_grid1 import AxesGrid
import matplotlib as mpl
import xarray as xr
import cartopy.crs as ccrs
from pyproj import CRS, Transformer

# Check Sample Size

### Warm seasons only (spring, summer, fall)

In [ ]:
# load fire observations
fire_obs = gpd.read_file('/net/rain/hyclimm/data/projects/SynFire/WP1/Regionalization/FRYv2.0_FireCCI51_6D_2001-2020_study_area_fire_observations_w_region.shp')
fire_obs["start_date"] = pd.to_datetime(fire_obs["start_date"])

In [ ]:
for reg in ["BI", "IP", "FR", "ME", "AL", "SEA", "NEA", "SC", "WMD", "EMD"]:

    # get fire_days and nofire_days as a list
    all_days = [date for date in pd.date_range(start = "2001-01-01", end = "2020-12-31") if date.month in [3, 4, 5, 6, 7, 8, 9, 10, 11]]
    fire_days = [date for date in set(fire_obs[fire_obs["region"] == reg]["start_date"]) if date.month in [3, 4, 5, 6, 7, 8, 9, 10, 11]]  #unique days with fires
    nofire_days = [date for date in all_days if date not in fire_days]

    #stratify fire_days into regfire and synfire
    regfire_days = [date for date in fire_days if set({reg}) == set(fire_obs[fire_obs["start_date"] == date]['region'])]
    synfire_days = [date for date in fire_days if date not in regfire_days]

    print(reg, len(nofire_days), len(regfire_days), len(synfire_days), len(fire_days), len(regfire_days)+len(synfire_days), len(all_days), len(nofire_days)+len(regfire_days)+len(synfire_days))

### All seasons

In [ ]:
for reg in ["BI", "IP", "FR", "ME", "AL", "SEA", "NEA", "SC", "WMD", "EMD"]:

    # get fire_days and nofire_days as a list
    all_days = list(pd.date_range(start = "2001-01-01", end = "2020-12-31"))
    fire_days = list(set(fire_obs[fire_obs["region"] == reg]["start_date"]))  #unique days with fires
    nofire_days = [date for date in all_days if date not in fire_days]

    #stratify fire_days into regfire and synfire
    regfire_days = [date for date in fire_days if set({reg}) == set(fire_obs[fire_obs["start_date"] == date]['region'])]
    synfire_days = [date for date in fire_days if date not in regfire_days]

    print(reg, len(nofire_days), len(regfire_days), len(synfire_days), len(fire_days), len(regfire_days)+len(synfire_days), len(all_days), len(nofire_days)+len(regfire_days)+len(synfire_days))

# Compare for each region

In [ ]:
def plot_no_reg_syn_fires(reg):
    
    #-------------------------------
    #projection
    lcc_crs = ccrs.LambertConformal(
            central_longitude=8, central_latitude=50, 
            standard_parallels=(50, 50),  
            false_easting=2937000.506058639, 
            false_northing=2937000.470434457
        )
    
    #-------------------------------
    #layout
    fig, g_axes = plt.subplots(nrows = 5, ncols = 3, subplot_kw = dict(projection = lcc_crs), figsize = (10, 12), constrained_layout = True)
    g_axes = g_axes.flatten()
    
    #-------------------------------
    #colorlist
    
    #BuRd 10 (for mx2t, 10si, fwi)
    colorlist1 = ["#053061", "#2166ac", "#4393c3", "#92c5de", "#d1e5f0", "#fddbc7", "#f4a582", "#d6604d", "#b2182b", "#67001f"]
    cmap1 = ListedColormap(colorlist1)
    bounds1 = [-0.75, -0.6, -0.45, -0.3, -0.15, 0, 0.15, 0.3, 0.45, 0.6, 0.75] #len(bounds) = len(colors) + 1
    norm1 = BoundaryNorm(bounds1, cmap1.N)
    
    #RdBu 10 (for tp, 2r)
    colorlist2 = colorlist1[::-1]
    cmap2 = ListedColormap(colorlist2)
    bounds2 = [-0.75, -0.6, -0.45, -0.3, -0.15, 0, 0.15, 0.3, 0.45, 0.6, 0.75] #len(bounds) = len(colors) + 1
    norm2 = BoundaryNorm(bounds2, cmap2.N)
    
    #label
    label_dict = dict(zip(["mx2t", "10si", "tp", "2r", "fwi"], ["Maximum Temperature", "Wind Speed", "Total Precipitation", "Mean Relative Humidity", "Fire Weather Index"]))
    sub_lb_dict = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)","(g)", "(h)", "(i)","(j)", "(k)", "(l)", "(m)", "(n)", "(o)"]
    
    
    #-------------------------------
    i = 0
    
    for varname in ["mx2t", "10si", "tp", "2r", "fwi"]:
    
        if varname in ["mx2t", "10si", "fwi"]:
            cmap, bounds, norm = cmap1, bounds1, norm1
        else:
            cmap, bounds, norm = cmap2, bounds2, norm2
    
        for cl in ['nofire', 'regfire', 'synfire']:
    
            ax = g_axes[i]
    
            var = xr.open_dataarray(f'/net/rain/hyclimm/data/projects/SynFire/WP1/No_vs_Reg_vs_Syn_fires_monthly_standardized_anomaly/{reg}_{varname}_mean_anomaly_{cl}.nc')
            var = var.rio.write_crs(lcc_crs)
    
            im = var.plot(
                ax = ax,
                transform = lcc_crs,
                cmap = cmap,
                norm = norm,
                add_colorbar = False
            )
            ax.coastlines()
    
            
            #add colorbar
            if i%3 == 2:
                cbar = plt.colorbar(im, ax = ax, extend='both', boundaries=bounds, ticks=bounds, location = 'right')
                cbar.set_label('Standardized Anomaly', fontsize = 12)
                cbar.ax.set_yticks([-0.75, -0.6, -0.45, -0.3, -0.15, 0, 0.15, 0.3, 0.45, 0.6, 0.75])
                cbar.ax.set_yticklabels(["-0.75", "-0.60", "-0.45", "-0.30", "-0.15", "0", "0.15", "0.30", "0.45", "0.60", "0.75"])
            
            #add y label
            if not i%3:
                ax.text(-0.15, 0.5, label_dict[varname], va='center', rotation='vertical', transform=ax.transAxes, fontsize=13) #ax.set_ylabel() does not work for a geo_axis
    
            #add season label
            if varname == "mx2t":
                ax.set_title("No Fire" if i == 0 else "Regional Fire(s)" if i == 1 else "Synchronous Fires", fontsize = 13)
            else:
                ax.set_title("")
    
            #add subpanel label
            ax.text(0.03, 0.9, sub_lb_dict[i], transform = ax.transAxes, fontsize = 16, color = '#00008B')
            
            i += 1

    plt.savefig(f"/home/lixinh/SynFire_Scripts/WP1/Figures/Maps_no_reg_syn_fires_monthly_standardized_CERRA_vars_for_{reg}.png", dpi = 600, bbox_inches = "tight")

In [ ]:
for reg in ['SC', 'IP', 'BI', 'NEA', 'SEA', 'AL', 'WMD', 'EMD', 'ME', 'FR']:
    plot_no_reg_syn_fires(reg)

# Spatial Average (latitude weighted)

In [ ]:
#get spatial mean and standard deviation
rows = []

for reg in ['SC', 'IP', 'BI', 'NEA', 'SEA', 'AL', 'WMD', 'EMD', 'ME', 'FR']:

    for cl in ['nofire', 'regfire', 'synfire']:
        
        val  = dict()

        for varname in ['2t', 'mx2t', '2r', 'tp', '10si', 'fwi']:

            var = xr.open_dataarray(f'/net/rain/hyclimm/data/projects/SynFire/WP1/No_vs_Reg_vs_Syn_fires_monthly_standardized_anomaly/{reg}_{varname}_mean_anomaly_{cl}.nc')

            #-----------------------------
            #add lon, lat grids
            lcc_crs = {'grid_mapping_name': 'lambert_conformal_conic',
                       'standard_parallel': 50.0,
                       'longitude_of_central_meridian': 8.0,
                       'latitude_of_projection_origin': 50.0,
                       'earth_radius': 6371229.0,
                       'false_easting': 2937000.506058639,
                       'false_northing': 2937000.470434457,
                       'longitudeOfFirstGridPointInDegrees': 342.514057,
                       'latitudeOfFirstGridPointInDegrees': 20.292281,
                      }
            
            crs_proj = CRS.from_cf(lcc_crs)
            crs_geo = CRS.from_epsg(4326)
            transformer = Transformer.from_crs(crs_from = crs_proj, crs_to = crs_geo, always_xy=True)
            
            x = var['x'].values
            y = var['y'].values
            
            xx, yy = np.meshgrid(x, y)  #inputs of length M and N, the outputs are of shape (N, M) by default
            
            lon, lat = transformer.transform(xx, yy)
            
            var = var.assign_coords(
            lon = (('y', 'x'), lon),
            lat = (('y', 'x'), lat)
            )
            
            #calculate weight
            weights = np.cos(np.deg2rad(var.lat))
            weights.name = 'weights'
            
            #weighted mean
            var_weighted_mean = var.weighted(weights).mean().values
            
            #weighted standard deviation
            var_weighted_variance = (
                (var - var_weighted_mean)**2
            ).weighted(weights).mean()
            
            var_weighted_std = np.sqrt(var_weighted_variance).values
            #-----------------------------

            val.update({f'{varname}_mean': var_weighted_mean, f'{varname}_std': var_weighted_std})

        
        val.update({'type':cl, 'reg': reg})
        rows.append(val)
        
df = pd.DataFrame(rows)

In [ ]:
#save
df.to_csv("/net/rain/hyclimm/data/projects/SynFire/WP1/No_vs_Reg_vs_Syn_fires_monthly_standardized_anomaly/No_vs_Reg_vs_Syn_fires_regional_mean_std_lat_weighted.csv", index = False)

In [ ]:
#plot
df = pd.read_csv("/net/rain/hyclimm/data/projects/SynFire/WP1/No_vs_Reg_vs_Syn_fires_monthly_standardized_anomaly/No_vs_Reg_vs_Syn_fires_regional_mean_std_lat_weighted.csv")

fig, axes = plt.subplots(2, 3, figsize = (15, 15))
axes = axes.flatten()
axes[-1].set_axis_off()

title_dict = {
    'mx2t': '(a) Maximum Temperature',
    '2r': '(b) Mean Relative Humidity',
    'tp': '(c) Total Precipitation',
    '10si': '(d) Mean Wind Speed',
    'fwi': '(e) Fire Weather Index'
}

for ax, var in zip(axes, ['mx2t', '2r', 'tp', '10si', 'fwi']):

    dfsub = df[[f'{var}_mean', f'{var}_std', 'type', 'reg']]
    dfsub = dfsub[dfsub['type'].isin(['regfire', 'synfire'])]

    for i, reg in enumerate(['IP', 'WMD', 'EMD', 'FR', 'AL', 'SEA', 'BI', 'ME', 'NEA', 'SC']):

        x1 = dfsub[(dfsub['reg'] == reg) & (dfsub['type'] == 'regfire')][f'{var}_mean']
        x1err = dfsub[(dfsub['reg'] == reg) & (dfsub['type'] == 'regfire')][f'{var}_std']

        x2 = dfsub[(dfsub['reg'] == reg) & (dfsub['type'] == 'synfire')][f'{var}_mean']
        x2err = dfsub[(dfsub['reg'] == reg) & (dfsub['type'] == 'synfire')][f'{var}_std']

        ax.errorbar(x = x1, xerr = x1err, y = i-0.1, color = '#bf0a30', fmt = 'o')
        ax.errorbar(x = x2, xerr = x2err, y = i+0.1, color = '#420d09', fmt = 'o')

    ax.set_xlim(-1.2, 1.2)
    ax.set_xlabel('Standardized Anomaly (-)', fontsize = 13)
    ax.set_title(title_dict[var], fontsize = 15)
    ax.axvline(x = 0, color = '#dadada', linestyle = '--')
    ax.spines["left"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis = 'x', labelsize = 12)
    
    #Set region names
    ax.set_yticks([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

    # ME->CE, NEA->NEE, SEA->SEE
    ax.set_yticklabels(['IP', 'WMD', 'EMD', 'FR', 'AL', 'SEE', 'BI', 'CE', 'NEE', 'SC'], fontsize = 13)
 

lg_reg = mlines.Line2D([0], [0], marker='o', color='#bf0a30', label='Regional Fire(s)', markersize=6)
lg_syn = mlines.Line2D([0], [0], marker='o', color='#420d09', label='Synchronous Fires', markersize=6)

plt.legend(handles = [lg_syn, lg_reg], ncol=1, frameon = False, fontsize = 15, bbox_to_anchor = (0.9, 0.65), labelspacing = 2)
plt.subplots_adjust(wspace = 0.2)

plt.savefig("/home/lixinh/SynFire_Scripts/WP1/Figures/v5_3_weather_conditions.png", dpi = 600, bbox_inches = 'tight')